# Exercise 00
## PyTorch Introduction - Part III: Training a Network

### Goals of this notebook

- Get an overview of the components of a deep learning pipeline.
- Write your own `Dataset` that loads images from disk, and use it with a `DataLoader`.
- Define your own network and implement the training step.
- Evaluate the network and submit your code and model to the test server.

## Setup

On Google Colab, the first cell mounts your Google Drive. We presume that you uploaded the `cv3dst` folder to the top level of your drive. The shared folders for datasets and outputs are located relative to the `cv3dst` folder.

Datasets with many small files are very slow to read from a mounted Google Drive. On Colab, we therefore write the PNG files of this notebook to the local disk of the Colab machine (`/content/datasets`), while MNIST itself is read from the shared `datasets` folder. The PNG files are deleted when the Colab session ends and written again in the next session.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, TensorDataset, random_split

USING_COLAB = "google.colab" in sys.modules

if USING_COLAB:
    import os
    from google.colab import drive

    gdrive_path = "/content/gdrive/MyDrive/cv3dst/exercise_00"

    # This will mount your google drive under 'MyDrive'
    drive.mount("/content/gdrive", force_remount=True)
    # Navigate to the exercise folder and check manually that all files are present
    os.chdir(gdrive_path)
    print(sorted(os.listdir()))

root_dir = Path.cwd().parent  # the cv3dst folder
dataset_dir = root_dir / "datasets"
png_dir = Path("/content/datasets") if USING_COLAB else dataset_dir  # where the PNG export goes, see above
output_dir = root_dir / "exercise_00" / "models"
output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from exercise_code import MNISTDataset, Net, evaluate, export_mnist_png, train
from exercise_code.data.dataset import MEAN, STD
from exercise_code.test import test_mnist_dataset, test_network, test_one_epoch, test_train_step

## The deep learning pipeline

Almost every supervised deep learning project consists of the same components:

```
 files on disk ──► Dataset ──────► DataLoader ──► model ──► loss ──────► optimizer
 images, labels    one sample      batches        forward   compare the   update the
                   + transforms    + shuffling    pass      prediction    weights
                                                            to the label
```

PyTorch provides a building block for each component, so you only write the parts that are specific to your problem. In this notebook, we train a network to classify the handwritten digits of the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) dataset: grayscale images of size $28 \times 28$ that show one of the digits 0 to 9.

| Component | In this notebook |
|---|---|
| Dataset | `MNISTDataset` in `exercise_code/data/dataset.py`, **your task** |
| DataLoader | `DataLoader` with `shuffle=True` for training |
| Model | `Net` in `exercise_code/data/network.py`, **your task** |
| Loss | `nn.CrossEntropyLoss` |
| Optimizer | `optim.SGD` with momentum |
| Training step | `train_step` in `exercise_code/data/train.py`, **your task** |

## 1. Why not do it by hand?

Training works on small random subsets of the data, called *mini-batches*. If all images fit into memory as one tensor, we could create the batches by hand:

In [ ]:
X = torch.randn(1000, 1, 28, 28)  # pretend these are 1000 images in memory
y = torch.randint(0, 10, (1000,))

batch_size = 64
perm = torch.randperm(len(X))  # new random order in every epoch
for start in range(0, len(X), batch_size):
    idx = perm[start : start + batch_size]
    X_batch, y_batch = X[idx], y[idx]
    # ... one training step with X_batch, y_batch ...
print(X_batch.shape, y_batch.shape)

This does not scale to real problems:
- Image datasets do not fit into memory. Every sample must be loaded from disk and decoded when needed.
- Samples often need preprocessing and random augmentation.
- Loading should run in parallel to the training, so the GPU does not wait for data.

As in the first notebook, the answer is to use the existing building blocks. With a `Dataset` you only describe how to get **one** sample; the `DataLoader` handles shuffling, batching and parallel loading.

## 2. The `Dataset` class

A custom dataset inherits from `torch.utils.data.Dataset` and implements three methods:
- `__init__` runs once and prepares everything that is needed to access the samples, e.g. a list of file names.
- `__len__` returns the number of samples.
- `__getitem__(index)` returns the sample at position `index`, e.g. as a tuple `(input, target)`.

For images, `__init__` only collects the file names and labels, which is cheap. `__getitem__` loads a single image from disk when it is requested. This way, the dataset never holds all images in memory.

### The data on disk

The next cell loads MNIST from the `datasets` folder (and downloads it if it is missing) and writes it in the typical layout of an image dataset: one PNG file per image and a `labels.csv` file with the file name and label of every image, separately for the training and the test split. This takes a minute, but only the first time.

In [ ]:
mnist_dir = export_mnist_png(dataset_dir, png_dir)
print(sorted(p.name for p in mnist_dir.iterdir()))
print((mnist_dir / "train" / "labels.csv").read_text().splitlines()[:3])

<div class="alert alert-block alert-warning"><b>Note:</b> MNIST is tiny. All 70,000 images need only about 55 MB as <code>uint8</code> and would easily fit into memory. Loading them all in <code>__init__</code> would even be faster here. We still load them lazily from disk, because this is how you handle the datasets of detection, segmentation and tracking, e.g. the thousands of Full HD video frames of a tracking benchmark.</div>

### Your dataset

`__init__` of `MNISTDataset` is already implemented: it reads `labels.csv` into a list of file names and a list of labels. `__getitem__` should return the image as a normalized float tensor:
- PNG files contain `uint8` values in `[0, 255]`. Remember the `uint8` tip from notebook 1: convert to `float32` and scale to `[0, 1]` first.
- Networks train better if their inputs have zero mean and unit variance. We normalize the images with the mean and standard deviation of the MNIST training images, `(img - MEAN) / STD`.

The dataset takes an optional `transform`, a function that is applied to every loaded sample. Transforms are also the place for random data augmentation. [`torchvision.transforms`](https://pytorch.org/vision/stable/transforms.html) provides many of them.

<div class="alert alert-info">
    <h3>Task: <code>MNISTDataset</code></h3>
    <p>Implement <code>__len__</code> and <code>__getitem__</code> of <code>MNISTDataset</code> in <a href="exercise_code/data/dataset.py"><code>exercise_code/data/dataset.py</code></a>. Open the image file with <a href="https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.open"><code>PIL.Image.open</code></a> and convert it to a tensor with <a href="https://pytorch.org/vision/stable/generated/torchvision.transforms.functional.pil_to_tensor.html"><code>torchvision.transforms.functional.pil_to_tensor</code></a>.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: <code>MNISTDataset</code></h3>
    <p>Run the following cell to test your dataset on a few random images. The test server runs the same checks with other images.</p>
</div>

In [ ]:
_ = test_mnist_dataset()

In [ ]:
train_val_data = MNISTDataset(mnist_dir / "train")
test_data = MNISTDataset(mnist_dir / "test")
img, label = train_val_data[0]
print(f"{len(train_val_data)} training and {len(test_data)} test images")
print(f"image: {tuple(img.shape)} {img.dtype}, mean {img.mean():.2f}, label: {label}")


def show_images(images, titles, colors=None):
    fig, axs = plt.subplots(2, 8, figsize=(16, 5))
    for i, ax in enumerate(axs.flat):
        ax.imshow(images[i, 0].cpu(), cmap="gray")  # (1, H, W) -> (H, W)
        ax.set_title(titles[i], color=colors[i] if colors else "black")
        ax.axis("off")
    plt.show()


samples = [train_val_data[i] for i in range(16)]
show_images(torch.stack([img for img, _ in samples]), [str(label) for _, label in samples])

<div class="alert alert-block alert-info"><b>Use what exists:</b> for images sorted into one folder per class, <a href="https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html"><code>torchvision.datasets.ImageFolder</code></a> does this for you. torchvision also ships many common datasets, including MNIST.</div>

## 3. Splitting a dataset

We use a *validation* split of the training data to check the network during development, and the test split only at the end. `random_split` accepts absolute lengths or fractions. A fixed `generator` seed produces the same split in every run, so validation samples never end up in the training set.

In [ ]:
train_data, val_data = random_split(train_val_data, [0.9, 0.1], generator=torch.Generator().manual_seed(0))
print(f"{len(train_data)} training and {len(val_data)} validation images")
print(type(train_data), train_data.dataset is train_val_data, train_data.indices[:5])

The splits are `Subset` objects. A `Subset` does not copy any data: it stores the original dataset (`.dataset`) and the selected indices (`.indices`). Indexing a subset works just like indexing the dataset.

## 4. The `DataLoader`

A [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) wraps a dataset and returns batches. The most important arguments are:
- `batch_size`: the number of samples per batch.
- `shuffle`: draw the samples in a new random order in every epoch. Use `True` for training and `False` for evaluation.
- `drop_last`: drop the last batch if it is smaller than `batch_size`.
- `num_workers`: the number of background processes that load data in parallel. The default `0` loads in the main process.
- `pin_memory`: load the batches into pinned memory, which speeds up the copy to the GPU (see notebook 2).

The `DataLoader` calls `__getitem__` for every sample of a batch and stacks the results along a new first dimension, the *batch dimension*. One pass over the `DataLoader` is one **epoch**.

In [ ]:
batch_size = 64
loader_args = {"batch_size": batch_size, "num_workers": 2, "pin_memory": device.type == "cuda"}
train_loader = DataLoader(train_data, shuffle=True, **loader_args)
val_loader = DataLoader(val_data, **loader_args)
test_loader = DataLoader(test_data, **loader_args)

images, labels = next(iter(train_loader))
print(f"images: {tuple(images.shape)} {images.dtype}, labels: {tuple(labels.shape)} {labels.dtype}")
print(f"{len(train_data)} samples -> {len(train_loader)} batches, the last one with {len(train_data) % batch_size} samples")
show_images(images, [str(label.item()) for label in labels])

How much does loading from disk cost? We time the first 200 batches when loading lazily from disk, with and without background workers, and from a dataset that holds all images in memory.

In [ ]:
def time_batches(loader, num_batches=200):
    start = time.perf_counter()
    for _, _ in zip(range(num_batches), loader):
        pass
    return time.perf_counter() - start


raw = torchvision.datasets.MNIST(dataset_dir, train=True)  # the images as one uint8 tensor
in_memory = TensorDataset((raw.data[:, None].float() / 255 - MEAN) / STD, raw.targets)

print(f"from disk, num_workers=0: {time_batches(DataLoader(train_val_data, batch_size=64, shuffle=True)):.2f} s")
print(f"from disk, num_workers=2: {time_batches(DataLoader(train_val_data, batch_size=64, shuffle=True, num_workers=2)):.2f} s")
print(f"in memory:                {time_batches(DataLoader(in_memory, batch_size=64, shuffle=True)):.2f} s")

Background workers hide most of the loading time, as long as the training step takes longer than loading a batch. The workers need a moment to start, which is included in the measurement.

## 5. The network

A network in PyTorch is a class that inherits from `nn.Module`:
- `__init__` creates the layers.
- `forward` defines how the input flows through the layers.

The network returns one raw score per class, the so-called *logits*. Do not apply a softmax: `nn.CrossEntropyLoss` does this internally.

We suggest two convolutional networks. Both turn the image of shape `(B, 1, 28, 28)` into a feature map and end with a *head*, which turns the feature map into the logits `(B, 10)`.

**A: a small network with a flatten head**

`Conv2d(1, 16, 3, padding=1)`, `ReLU`, `MaxPool2d(2)`, `Conv2d(16, 32, 3, padding=1)`, `ReLU`, `MaxPool2d(2)` → feature map `(B, 32, 7, 7)` → `Flatten` → `Linear(32 · 7 · 7, 10)`

The head keeps the spatial layout: every position of the feature map has its own weights. With 20,490 parameters, 77 % of them in the head, A reaches about 98 % test accuracy after two epochs. It only works for a fixed input size.

**B: a residual network with a global pooling head**

| Stage | Layers | Output |
|---|---|---|
| stem | `Conv2d(1, 16, 3, padding=1)`, `BatchNorm2d(16)`, `ReLU` | `(B, 16, 28, 28)` |
| stage 1 | residual block with 16 channels | `(B, 16, 28, 28)` |
| downsampling | `Conv2d(16, 32, 3, stride=2, padding=1)`, `BatchNorm2d(32)`, `ReLU` | `(B, 32, 14, 14)` |
| stage 2 | residual block with 32 channels | `(B, 32, 14, 14)` |
| downsampling | `Conv2d(32, 64, 3, stride=2, padding=1)`, `BatchNorm2d(64)`, `ReLU` | `(B, 64, 7, 7)` |
| stage 3 | residual block with 64 channels | `(B, 64, 7, 7)` |
| head | `AdaptiveAvgPool2d(1)`, `Flatten`, `Linear(64, 10)` | `(B, 10)` |

A *residual block* with $C$ channels computes $\text{ReLU}(x + f(x))$, where $f$ is `Conv2d(C, C, 3, padding=1)`, `BatchNorm2d(C)`, `ReLU`, `Conv2d(C, C, 3, padding=1)`, `BatchNorm2d(C)`. Write it as its own `nn.Module`; the addition of the input $x$ is the *skip connection*.

- The global pooling head averages the feature map over all positions. It needs only 650 parameters and works for any input size, but it discards *where* a feature occurs. The network therefore needs enough layers before the pooling, so that every position of the final feature map sees a large part of the digit.
- Deep networks are hard to train. Without the skip connections, the same network often does not learn at all with our optimizer. The skip connections let the gradient flow directly to the early layers.
- The network has 121,610 parameters, most of them in the convolutions with 64 channels: a convolution has $3 \cdot 3 \cdot C_{in} \cdot C_{out}$ weights. It trains slower per epoch than A, but reaches about 99 % test accuracy.
- Batch normalization behaves differently in training and evaluation mode (notebook 2, section 4). `evaluate` switches the network to `eval()` for you.

`nn.Sequential` chains layers, so you do not need to call each of them in `forward`.

<div class="alert alert-info">
    <h3>Task: <code>Net</code></h3>
    <p>Implement <code>__init__</code> and <code>forward</code> of <code>Net</code> in <a href="exercise_code/data/network.py"><code>exercise_code/data/network.py</code></a>. Use one of the suggested architectures or your own, with at most 500,000 parameters. The test server creates your network with <code>Net()</code>.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: <code>Net</code></h3>
    <p>Run the following cell to test the output shape and the number of parameters of your network.</p>
</div>

In [ ]:
_ = test_network()

<div class="alert alert-block alert-info">
Thanks to <b>autograd</b>, we only have to define the <b>forward()</b> function. PyTorch derives the backward pass, which computes the gradients, automatically.
</div>

`print(net)` lists the layers of a network. `net.parameters()` returns all learnable parameters, which is what we pass to the optimizer. Like tensors, the network has to be moved to the `device`.

In [ ]:
torch.manual_seed(0)
net = Net().to(device)
print(net)

for name, parameter in net.named_parameters():
    print(f"{name:<20} {tuple(parameter.shape)}")
print(f"total: {sum(p.numel() for p in net.parameters()):,} parameters")

## 6. Loss function and optimizer

This is a multi-class classification problem, so we use the cross-entropy loss. We optimize it with stochastic gradient descent (SGD) with momentum. The modules `torch.nn` and `torch.optim` contain many more loss functions and optimizers.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)

## 7. Training the network

Training repeats the same *training step* for every batch:

1. Move the batch to the device of the network.
2. `optimizer.zero_grad()`: reset the gradients of all parameters. Remember from notebook 2 that gradients accumulate otherwise.
3. `y_pred = model(X)`: the forward pass computes the logits.
4. `loss = criterion(y_pred, y)`: compare the predictions to the labels.
5. `loss.backward()`: the backward pass computes the gradients of the loss with respect to all parameters.
6. `optimizer.step()`: update the parameters with the gradients.

The function `train` in the same file loops over the epochs and batches, calls `train_step` and tracks the average loss and accuracy of the last `log_every` batches.

<div class="alert alert-info">
    <h3>Task: <code>train_step</code></h3>
    <p>Implement <code>train_step</code> in <a href="exercise_code/data/train.py"><code>exercise_code/data/train.py</code></a>.</p>
</div>

<div class="alert alert-danger">
    <h3>Test: <code>train_step</code></h3>
    <p>The first test compares three training steps of your function with a reference implementation. The second test trains a new instance of your network for one epoch with your dataset and your training step, and checks that the loss on the test set goes down and the accuracy goes up.</p>
</div>

In [ ]:
_ = test_train_step()

In [ ]:
_ = test_one_epoch(train_loader, test_loader, device)

Now we train the network for real.

In [ ]:
num_epochs = 2
log_every = 100
loss_history, acc_history = train(net, train_loader, optimizer, criterion, num_epochs, device, log_every)

In [ ]:
iterations = [log_every * (i + 1) for i in range(len(loss_history))]

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
axs[0].plot(iterations, loss_history)
axs[0].set(title="training loss", xlabel="iteration", ylabel="loss")
axs[1].plot(iterations, acc_history)
axs[1].set(title="training accuracy", xlabel="iteration", ylabel="accuracy")
plt.show()

## 8. Evaluating the network

For evaluation, we
- switch the network to evaluation mode with `net.eval()`. Our network behaves the same in both modes, but layers such as dropout and batch normalization do not,
- disable gradient tracking with `torch.no_grad()`,
- take the class with the highest score as the prediction with `argmax`.

`evaluate` in `exercise_code/data/train.py` does this and returns the average loss and the accuracy.

In [ ]:
val_loss, val_acc = evaluate(net, val_loader, criterion, device)
print(f"Validation loss: {val_loss:.3f}, accuracy: {100 * val_acc:.1f} %")

Once you are happy with the validation accuracy, evaluate the network on the test set. The test server computes the same test accuracy for your saved model: it gives full points from 97 %. The function below collects all predictions. We only loop over the batches, which cannot be avoided; everything else is vectorized. Let's look at some test images with the prediction and, in brackets, the true label. Correct predictions are green, wrong ones red.

In [ ]:
def predict(net, loader):
    net.eval()
    preds, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            preds.append(net(images.to(device)).argmax(dim=1).cpu())
            labels.append(y)
    net.train()
    return torch.cat(preds), torch.cat(labels)


preds, labels = predict(net, test_loader)
correct = preds == labels
print(f"Test accuracy: {100 * correct.float().mean():.1f} %")

images, _ = next(iter(test_loader))
titles = [f"{p}\n({l})" for p, l in zip(preds[:16].tolist(), labels[:16].tolist())]  # the test loader is not shuffled
show_images(images, titles, colors=["green" if c else "red" for c in correct[:16]])

Which digits does the network recognize well? Remember `torch.bincount` from the first notebook: counting the labels of all test images and of the correctly classified images gives the accuracy per class, without a loop over the samples.

In [ ]:
class_total = torch.bincount(labels, minlength=10)
class_correct = torch.bincount(labels[correct], minlength=10)
class_acc = class_correct / class_total

for digit, (acc, total) in enumerate(zip(class_acc, class_total)):
    print(f"digit {digit}: {100 * acc:5.1f} % of {total} images")

## 9. Saving the model and submitting

We save the learned parameters (the *state dict*) of the network to the `models` folder. Then we create a zip file with your code and the model for the upload to the test server. The server grades
- your tensor operations from notebook 1,
- your dataset, your network and your training step,
- the test accuracy of your saved model.

Make sure that the saved model and the definition of `Net` in your python file match.

In [ ]:
torch.save(net.state_dict(), output_dir / "model.pt")

In [ ]:
from exercise_code.submit import submit_exercise

submit_exercise(str(root_dir / "output" / "exercise00"))

## References

1. [PyTorch Tutorial: Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
2. [PyTorch Tutorial: Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
3. [PyTorch Tutorial: Optimizing Model Parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)